In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
display(dbutils.fs.ls(silver_path))

In [0]:
%python
#Tabelas temporarias
silver_mapeamento= {
    'temp_silver_clientes' : f'{silver_path}/clientes/',
    'temp_silver_itens_pedido' : f'{silver_path}/itens_pedido/',
    'temp_silver_pedidos' : f'{silver_path}/pedidos/',
    'temp_silver_produtos' : f'{silver_path}/produtos/',
    'temp_silver_vendedores' : f'{silver_path}/vendedores/'

}
for view_name, path in silver_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
%sql
select * from temp_silver_itens_pedido

In [0]:
%sql
describe temp_silver_clientes

In [0]:
%python
#enriquecendo os dados de pedidos, clientes, vendedores, itens e produtos por meio de JOINs, calcula o valor total de cada item e salva o resultado como Delta na camada Silver para visualização 
pedidos_enriquecidos = spark.sql("""
WITH clientes_silver AS (

    SELECT
        id_cliente,
        nome,
        email,
        estado,
        data_cadastro,
        status_cliente

    FROM temp_silver_clientes

),

produtos_silver AS (
       SELECT
        id_produto,
        nome_produto,
        categoria,
        preco_unitario,
        marca,
        status

    FROM temp_silver_produtos
),

   
vendedores_silver AS (

   SELECT
        id_vendedor,
        nome_vendedor,
        estado,
        regiao,
        salario_base,
        data_contratacao,
        status
        

    FROM temp_silver_vendedores


),

pedidos_silver AS (

    SELECT DISTINCT
        id_pedido,
        id_cliente,
        id_vendedor,
        dataPedido,
        status_pedido,
        desconto

    FROM temp_silver_pedidos

  
),

itens_pedido_silver AS (

    SELECT
        id_item,
        id_pedido,
        id_produto,
        quantidade,
        marca

    FROM temp_silver_itens_pedido
)

SELECT
    p.id_pedido,
    p.dataPedido,
    p.status_pedido,
    p.desconto,

    c.id_cliente,
    c.nome AS nome_cliente,
    c.email,
    c.estado AS estado_cliente,
    c.data_cadastro,
    c.status_cliente,

    v.id_vendedor,
    v.nome_vendedor,
    v.estado AS estado_vendedor,
    v.regiao,
    v.salario_base,
    v.data_contratacao,
    v.status AS status_vendedor,

    i.id_item,
    i.id_produto,
    i.quantidade,
    i.marca,

    pr.nome_produto,
    pr.categoria,
    pr.preco_unitario,
    pr.marca AS marca_produto_catalogo,
    pr.status AS status_produto,

    ROUND(
        i.quantidade
        * pr.preco_unitario
        * (1 - p.desconto),
        2
    ) AS valor_total_item

FROM pedidos_silver p

INNER JOIN clientes_silver c
    ON p.id_cliente = c.id_cliente

INNER JOIN vendedores_silver v
    ON p.id_vendedor = v.id_vendedor

INNER JOIN itens_pedido_silver i
    ON p.id_pedido = i.id_pedido

INNER JOIN produtos_silver pr
    ON i.id_produto = pr.id_produto

""")

# Salvar em delta na silver
pedidos_enriquecidos.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeSchema', 'true')\
                .save(f'{silver_path}/pedidos_enriquecidos')

display(pedidos_enriquecidos)

In [0]:
%sql
--Criando tabela fisica
create table if not exists workspace.techvenda.pedidos_enriquecidos 
select * from delta. `/Volumes/workspace/techvenda/filestore/silver/pedidos_enriquecidos/`